<a href="https://colab.research.google.com/github/Youssif-Kady/flayrank_task1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import polars as pl
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, login

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

REPO_ID = "FlyRank/internship-warehouse"
api = HfApi(token=hf_token)
all_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
march_files = [f for f in all_files if "2026-03" in f and f.endswith(".parquet")]
if not march_files:
    march_files = [f for f in all_files if f.endswith(".parquet")][:5]

local_files = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=hf_token) for f in march_files]
df_march = pl.read_parquet(local_files)

df_model_data = df_march.group_by("content_hash_id").agg([
    pl.col("gsc_clicks").sum().alias("total_clicks"),
    pl.col("gsc_impressions").sum().alias("total_impressions"),
    pl.col("gsc_avg_position").mean().alias("avg_position"),
    pl.col("report_date").n_unique().alias("days_active")
]).filter(pl.col("total_impressions") > 0).to_pandas()


df_model_data['past_ctr'] = df_model_data['total_clicks'] / df_model_data['total_impressions']

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Label-derived features cause silent failures.

Methodology Question: Where does the label come from? Are any features in the dataset computed directly or indirectly from the target variable (e.g., computing a ratio using the target before splitting)?

Finding 2: Random splits are rarely honest.

Methodology Question: Does the validation design support the claim? Does a random split artificially inflate performance by allowing the model to memorize entities (like content_hash_id) across the train and test sets?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

features_leaky = ['total_impressions', 'avg_position', 'days_active', 'past_ctr']
X_leaky = df_model_data[features_leaky]
y = df_model_data['total_clicks']
groups = df_model_data['content_hash_id']

gkf = GroupKFold(n_splits=5)

print("=== Honest Split (GroupKFold) with Leaky Feature (past_ctr) ===")
fold_scores = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_leaky, y, groups=groups)):
    X_train, X_val = X_leaky.iloc[train_idx], X_leaky.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    score = r2_score(y_val, preds)
    fold_scores.append(score)

print(f"Average R2 Score (Still artificially high due to leakage): {np.mean(fold_scores):.4f}")

=== Honest Split (GroupKFold) with Leaky Feature (past_ctr) ===
Average R2 Score (Still artificially high due to leakage): 0.9097


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
from sklearn.metrics import mean_squared_error

# Removing the leaky feature 'past_ctr'
features_honest = ['total_impressions', 'avg_position', 'days_active']
X_honest = df_model_data[features_honest]

print("=== Leakage Audit: Honest Split WITHOUT Leaky Feature ===")
honest_fold_scores = []
honest_rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_honest, y, groups=groups)):
    X_train, X_val = X_honest.iloc[train_idx], X_honest.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    honest_fold_scores.append(r2_score(y_val, preds))
    honest_rmse_scores.append(np.sqrt(mean_squared_error(y_val, preds)))

print(f"Realistic Average R2 Score: {np.mean(honest_fold_scores):.4f}")
print(f"Realistic Average RMSE: {np.mean(honest_rmse_scores):.4f}")

=== Leakage Audit: Honest Split WITHOUT Leaky Feature ===
Realistic Average R2 Score: 0.4487
Realistic Average RMSE: 19.3429


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Claim (Week 5): The Random Forest Regressor drastically outperforms the Week-4 Baseline heuristic across all evaluation metrics (MAE down to 0.1665, R² improved to 0.9961).

Rewritten Safe Claim (Week 6): Upon auditing the model with a grouped split (GroupKFold on content_hash_id) and removing the label-derived feature past_ctr, the previously observed near-perfect R2 score collapsed to a realistic baseline (R2: 0.4487). The initial high performance was entirely due to data leakage. The corrected model provides directional decision-support based on impressions and ranking position, but limits in predictive power highlight the volatility of organic search traffic without historical CTR context.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.